### MVP Execution Order (Baseline SFT — Google Colab)
Run these cells in order for the baseline pass.

Connect to a Colab GPU runtime (T4/A100/H100), then run cells top-to-bottom.

**Notebook cells:**
1. Environment check (Colab detection, GPU info)
2. Install deps (`unsloth`, `transformers`, `trl`)
3. Model load (`FastLanguageModel.from_pretrained`)
4. LoRA config (`get_peft_model`)
5. Data Prep (Google Drive mount + schema preview + map)
6. Trainer setup
7. GPU memory snapshot
8. `trainer.train()`
9. Final memory/time stats
10. Primary inference sanity check
11. Save adapters locally + export to Google Drive

Notes:
- Dataset must be on Google Drive at `My Drive/irp/datasets/ir_playbooks_alpaca_v2_hybrid.jsonl`
- Adapters are exported to `My Drive/irp_exports/ir_assistant_lora/`
- If Data Prep fails (path/schema), fix that before running training.

## Environment Check

In [1]:
import os, sys

try:
    import torch  # type: ignore[reportMissingImports]
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "PyTorch is not installed in the active notebook kernel. "
        "Run `%pip install torch` (or use a Colab GPU runtime), then rerun this cell."
    ) from exc

# Compat patch: some torch builds expose `total_memory` (not `total_mem`)
if not getattr(torch.cuda.get_device_properties, "_compat_total_mem", False):
    _orig_get_device_properties = torch.cuda.get_device_properties

    class _CudaPropsCompat:
        def __init__(self, props):
            self._props = props

        @property
        def total_mem(self):
            return getattr(self._props, "total_mem", getattr(self._props, "total_memory"))

        def __getattr__(self, name):
            return getattr(self._props, name)

    def _get_device_properties_compat(device):
        props = _orig_get_device_properties(device)
        return props if hasattr(props, "total_mem") else _CudaPropsCompat(props)

    _get_device_properties_compat._compat_total_mem = True
    torch.cuda.get_device_properties = _get_device_properties_compat

# Colab environment check
in_colab = "google.colab" in sys.modules or os.path.exists("/content")
print(f"Running in Colab: {in_colab}")
print(f"Python: {sys.executable}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}")
    print(f"VRAM: {gpu.total_mem / 1024**3:.1f} GB")
    print(f"Compute capability: {gpu.major}.{gpu.minor}")
else:
    raise RuntimeError("No CUDA GPU detected. Make sure you're using a GPU runtime.")

Running in Colab: True
Python: /usr/bin/python3
PyTorch: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB
Compute capability: 12.0


In [2]:
%%capture
%pip install unsloth
%pip install transformers==4.56.2
%pip install --no-deps trl==0.22.2

### Model Load

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.7: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

### LoRA Configuration

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.3.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


### Pre-run Checklist (Pass/Fail)
Check these before executing Data Prep and training:

- [ ] Dataset path exists: `datasets/ir_playbooks_alpaca_v2_hybrid.jsonl`
- [ ] Dataset schema has required keys: `instruction`, `input`, `output`
- [ ] Data Prep preview prints at least 5 rows without errors
- [ ] Model load settings are unchanged (`load_in_4bit=True`)
- [ ] LoRA config cell ran without errors
- [ ] GPU is visible before train (`torch.cuda.get_device_properties(0)` works)
- [ ] Output directory is writable (`outputs/` and `ir_assistant_lora/` can be created)

If any item fails, fix it before continuing to `trainer.train()`.

<a name="Data"></a>
### Data Prep
Format the Alpaca dataset into a training split and append EOS tokens so generation stops correctly.

Dataset file used in this notebook:
- `datasets/ir_playbooks_alpaca_v2_hybrid.jsonl` (194 rows: 137 rewritten base + 57 MITRE-grounded synthetic)

In [5]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

from pathlib import Path
from datasets import load_dataset
from google.colab import drive

drive.mount("/content/drive")

data_file = Path("/content/drive/MyDrive/irp/datasets/ir_playbooks_alpaca_v2_hybrid.jsonl")

if not data_file.exists():
    raise FileNotFoundError(
        f"Dataset not found: {data_file}\n"
        "Upload ir_playbooks_alpaca_v2_hybrid.jsonl to Google Drive at:\n"
        "  My Drive/irp/datasets/ir_playbooks_alpaca_v2_hybrid.jsonl"
    )

with data_file.open("r", encoding="utf-8") as handle:
    row_count = sum(1 for line in handle if line.strip())
print(f"{data_file.name}: {row_count} rows")

dataset = load_dataset("json", data_files=str(data_file), split="train")

required_keys = {"instruction", "input", "output"}
missing_keys = required_keys - set(dataset.column_names)
if missing_keys:
    raise ValueError(f"Dataset missing required keys: {sorted(missing_keys)}")

print(f"Loaded rows: {len(dataset)}")
for i, row in enumerate(dataset.select(range(min(5, len(dataset))))):
    print(f"Row {i}: keys={list(row.keys())}")
    print(f"  instruction: {str(row['instruction'])[:120]}...")
    print(f"  input: {str(row['input'])[:120]}...")
    print(f"  output: {str(row['output'])[:120]}...")

dataset = dataset.map(formatting_prompts_func, batched=True,)

Mounted at /content/drive
ir_playbooks_alpaca_v2_hybrid.jsonl: 194 rows


Generating train split: 0 examples [00:00, ? examples/s]

Loaded rows: 194
Row 0: keys=['instruction', 'input', 'output']
  instruction: Create a detailed incident response playbook for the following cyber incident....
  input: Incident type: Ransomware
Target asset: Windows AD Server
Detection source: EDR Alert - Suspicious File Encryption
Initi...
  output: Playbook:

Phase: Identification
Action: Triage alert, confirm IOC via edr-console, snapshot affected host on ws-corp-po...
Row 1: keys=['instruction', 'input', 'output']
  instruction: Create a detailed incident response playbook for the following cyber incident....
  input: Incident type: Data Breach
Target asset: Linux Web Server
Detection source: SIEM Alert - Anomalous Data Exfiltration
Ini...
  output: Playbook:

Phase: Identification
Action: Analyze siem-splunk-01 logs, identify compromised endpoints using edr-console
T...
Row 2: keys=['instruction', 'input', 'output']
  instruction: Create a detailed incident response playbook for the following cyber incident....
  input: Incident

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

### Train/Validation Split (Quick)
Create a small validation split for sanity-check evaluation during training.

In [6]:
test_size = 1 if len(dataset) < 20 else 0.05
split_dataset = dataset.train_test_split(test_size=test_size, seed=3407, shuffle=True)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Train rows: {len(train_dataset)}")
print(f"Validation rows: {len(eval_dataset)}")

Train rows: 184
Validation rows: 10


<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [7]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Full training run
                # max_steps = 60,  # Commented out for full run
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
        eval_strategy = "epoch",
        save_strategy = "no",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/184 [00:00<?, ? examples/s]

num_proc must be <= 10. Reducing num_proc to 10 for dataset of size 10.


Unsloth: Tokenizing ["text"] (num_proc=10):   0%|          | 0/10 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [8]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA RTX PRO 6000 Blackwell Server Edition. Max memory = 94.971 GB.
7.277 GB of memory reserved.


In [9]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 184 | Num Epochs = 1 | Total steps = 23
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.455900,0.363900


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [10]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

61.1976 seconds used for training.
1.02 minutes used for training.
Peak reserved memory = 16.73 GB.
Peak reserved memory for training = 9.453 GB.
Peak reserved memory % of max memory = 17.616 %.
Peak reserved memory for training % of max memory = 9.954 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [11]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Asset-aware inference test — model should reference specific hostnames/IPs
test_input = """Incident type: Ransomware
Target asset: Windows AD Server
Detection source: EDR Alert - Suspicious File Encryption
Initial vector: Email Attachment (Phishing)
Tactics and techniques:
- Initial Access: Phishing
- Execution: User Execution
- Impact: Data Encrypted for Impact
Severity: High
Tags: malware, lateral_movement, windows, ransomware, phishing

Organization assets:
- ws-corp-pool: Dell OptiPlex 7090 workstation, corporate (10.10.1.0/24), managed by CrowdStrike Falcon, 150 Windows 11 workstations, medium criticality
- srv-dc-01: Dell PowerEdge R750 server, datacenter (10.20.1.10), primary domain controller, managed by CrowdStrike Falcon, Primary AD DS, critical criticality
- srv-dc-02: Dell PowerEdge R750 server, datacenter (10.20.1.11), secondary domain controller, managed by CrowdStrike Falcon, Secondary AD DS, critical criticality
- edr-console: CrowdStrike Falcon edr, cloud (cloud), endpoint detection and response, Covers all endpoints and servers, critical criticality
- siem-splunk-01: Splunk Enterprise 9.2 siem, datacenter (10.20.2.10), centralized log aggregation and alerting, 90-day hot retention - 1-year warm, critical criticality
- fw-perimeter-01: Palo Alto PA-850 firewall, dmz (10.0.0.1), perimeter filtering, Managed via Panorama, critical criticality
- fw-internal-01: Palo Alto PA-450 firewall, datacenter (10.20.0.1), east-west segmentation, Segments corporate from datacenter, high criticality
- backup-veeam-01: Veeam Backup and Replication 12 backup, datacenter (10.20.4.10), primary backup server, Daily full backups - 4-hour RPO, critical criticality"""

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Create a detailed incident response playbook for the following cyber incident.",
        test_input,
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
result = tokenizer.batch_decode(outputs)[0]

# Print just the response portion
if "### Response:" in result:
    response = result.split("### Response:")[-1].strip()
    print("=== MODEL OUTPUT (Asset-Aware Test) ===")
    print(response)
    print("\n=== ASSET REFERENCE CHECK ===")
    hostnames = ["srv-dc-01", "srv-dc-02", "fw-perimeter-01", "fw-internal-01",
                 "siem-splunk-01", "edr-console", "backup-veeam-01", "ws-corp-pool"]
    found = [h for h in hostnames if h in response]
    print(f"Asset hostnames referenced: {len(found)}/{len(hostnames)} — {', '.join(found)}")
else:
    print(result)

=== MODEL OUTPUT (Asset-Aware Test) ===
Playbook:

Phase: Identification
Action: Analyze ws-corp-pool and siem-splunk-01 logs for suspicious file encryption, confirm via edr-console
Tools: CrowdStrike Falcon, Splunk Enterprise 9.2
Target response time (minutes): 15

Phase: Containment
Action: Isolate srv-dc-01 and srv-dc-02 from corporate and datacenter networks, block traffic at fw-perimeter-01 and fw-internal-01
Tools: Palo Alto PA-850, Palo Alto PA-450
Target response time (minutes): 10

Phase: Eradication
Action: Remove ransomware from ws-corp-pool and srv-dc-01, 02 using edr-console and YARA rules
Tools: CrowdStrike Falcon, YARA
Target response time (minutes): 60

Phase: Recovery
Action: Restore from backup-veeam-01, reimage ws-corp-pool and srv-dc-01, 02, monitor for reinfection
Tools: Veeam Backup and Replication 12, CrowdStrike Falcon
Target response time (minutes): 120

Phase: Lessons Learned
Action: Update incident response plans, train staff on phishing and ransomware preven

<a name="Save"></a>
### Save Adapters
Save the LoRA adapters locally for reuse in later inference or integration steps.

In [12]:
model.save_pretrained("ir_assistant_lora")  # Local saving
tokenizer.save_pretrained("ir_assistant_lora")
# model.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving

('ir_assistant_lora/tokenizer_config.json',
 'ir_assistant_lora/special_tokens_map.json',
 'ir_assistant_lora/tokenizer.json')

### End of Baseline Notebook
Baseline SFT flow is complete when adapter files are saved to `ir_assistant_lora/`.

Next phase (separate): integration testing with the chatbot repository and any required integration updates.

In [13]:
import os
print("cwd:", os.getcwd())
print("saved folder exists:", os.path.isdir("ir_assistant_lora"))
if os.path.isdir("ir_assistant_lora"):
    for name in sorted(os.listdir("ir_assistant_lora")):
        print("-", name)

cwd: /content
saved folder exists: True
- README.md
- adapter_config.json
- adapter_model.safetensors
- special_tokens_map.json
- tokenizer.json
- tokenizer_config.json


### Export Adapters to Google Drive
Mount Google Drive and copy both the adapter folder and a zip archive for persistent storage.

In [14]:
import os
import shutil
from pathlib import Path

source_dir = Path('ir_assistant_lora')
if not source_dir.exists():
    raise FileNotFoundError(f"Missing adapter directory: {source_dir}")

mount_root = Path('/content/drive')
mydrive_root = mount_root / 'MyDrive'

# Attempt mount only if not already mounted
if not mydrive_root.exists():
    try:
        from google.colab import drive
        drive.mount(str(mount_root), force_remount=True)
    except Exception as exc:
        print("Google Drive mount failed in this session.")
        print("Open this notebook in Colab browser and run this cell once to complete Drive auth.")
        print(f"Mount error: {exc}")

if mydrive_root.exists():
    drive_export_dir = mydrive_root / 'irp_exports'
    drive_export_dir.mkdir(parents=True, exist_ok=True)

    # Copy folder
    folder_target = drive_export_dir / 'ir_assistant_lora'
    if folder_target.exists():
        shutil.rmtree(folder_target)
    shutil.copytree(source_dir, folder_target)

    # Create and copy zip
    zip_path = Path(shutil.make_archive('ir_assistant_lora', 'zip', str(source_dir)))
    zip_target = drive_export_dir / zip_path.name
    shutil.copy2(zip_path, zip_target)

    print(f"Saved folder to: {folder_target}")
    print(f"Saved zip to: {zip_target}")
else:
    # Keep local runtime zip as fallback artifact
    zip_path = Path(shutil.make_archive('ir_assistant_lora', 'zip', str(source_dir)))
    print(f"Drive unavailable. Created local runtime zip: {zip_path}")

Saved folder to: /content/drive/MyDrive/irp_exports/ir_assistant_lora
Saved zip to: /content/drive/MyDrive/irp_exports/ir_assistant_lora.zip


## Automated Test Suite
Runs 10 prompts across 5 categories and scores each output against quality criteria:
- All 5 NIST phases present (Identification, Containment, Eradication, Recovery, Lessons Learned)
- Real tool names (not generic like "Incident Response Software")
- Defender perspective (not attacker kill chain)
- Not truncated (ends with EOS token or complete sentence)
- Specific actionable steps (not vague like "Restore systems")

In [15]:
import re, textwrap

# --- Model setup (reload from Drive export) ---
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/irp_exports/ir_assistant_lora",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

INSTRUCTION = "Create a detailed incident response playbook for the following cyber incident."

# --- Asset context blocks for asset-aware tests ---
ENDPOINT_NETWORK_ASSETS = (
    "\n\nOrganization assets:\n"
    "- ws-corp-pool: Dell OptiPlex 7090 workstation, corporate (10.10.1.0/24), managed by CrowdStrike Falcon, 150 Windows 11 workstations, medium criticality\n"
    "- srv-dc-01: Dell PowerEdge R750 server, datacenter (10.20.1.10), primary domain controller, managed by CrowdStrike Falcon, Primary AD DS, critical criticality\n"
    "- srv-dc-02: Dell PowerEdge R750 server, datacenter (10.20.1.11), secondary domain controller, managed by CrowdStrike Falcon, Secondary AD DS, critical criticality\n"
    "- edr-console: CrowdStrike Falcon edr, cloud (cloud), endpoint detection and response, Covers all endpoints and servers, critical criticality\n"
    "- siem-splunk-01: Splunk Enterprise 9.2 siem, datacenter (10.20.2.10), centralized log aggregation and alerting, 90-day hot retention - 1-year warm, critical criticality\n"
    "- fw-perimeter-01: Palo Alto PA-850 firewall, dmz (10.0.0.1), perimeter filtering, Managed via Panorama, critical criticality\n"
    "- fw-internal-01: Palo Alto PA-450 firewall, datacenter (10.20.0.1), east-west segmentation, Segments corporate from datacenter, high criticality\n"
    "- backup-veeam-01: Veeam Backup and Replication 12 backup, datacenter (10.20.4.10), primary backup server, Daily full backups - 4-hour RPO, critical criticality"
)

EMAIL_IDENTITY_ASSETS = (
    "\n\nOrganization assets:\n"
    "- srv-mail-01: Microsoft Exchange Online email_server, cloud (cloud), email and calendaring, Hybrid config - MX via cloud, high criticality\n"
    "- mfa-provider: Duo Security mfa, cloud (cloud), multi-factor authentication, Enforced for VPN - email - and admin consoles, critical criticality\n"
    "- pam-vault-01: CyberArk PAS pam, datacenter (10.20.1.50), privileged access management, All admin credentials vaulted here, critical criticality\n"
    "- srv-dc-01: Dell PowerEdge R750 server, datacenter (10.20.1.10), primary domain controller, managed by CrowdStrike Falcon, Primary AD DS, critical criticality\n"
    "- edr-console: CrowdStrike Falcon edr, cloud (cloud), endpoint detection and response, Covers all endpoints and servers, critical criticality\n"
    "- siem-splunk-01: Splunk Enterprise 9.2 siem, datacenter (10.20.2.10), centralized log aggregation and alerting, 90-day hot retention - 1-year warm, critical criticality\n"
    "- proxy-web-01: Zscaler ZIA proxy, corporate (cloud), web filtering and SSL inspection, All outbound HTTP/HTTPS routed through Zscaler, medium criticality"
)

# --- 12 Test Prompts across 6 categories ---
test_prompts = {
    # Core Playbook Generation (no assets — tests generic capability)
    "Core-1: Ransomware + SMB lateral":
        "Ransomware detected on multiple Windows servers in the finance department. Lateral movement suspected via SMB. Several workstations have encrypted files.",
    "Core-2: Phishing + credential harvest":
        "Phishing campaign targeting C-suite executives. Multiple users clicked malicious links. Credential harvesting page mimicking SSO portal detected.",
    "Core-3: SQL injection on prod DB":
        "SQL injection attack detected on production database server. WAF alerts show suspicious query strings. Potential data exfiltration of customer PII.",
    "Core-4: Insider data exfiltration":
        "Insider threat detected. An employee in engineering has been copying proprietary source code to personal USB drives and uploading to external cloud storage.",

    # MITRE ATT&CK Techniques
    "MITRE-1: PowerShell abuse (T1059.001)":
        "Incident type: PowerShell Abuse\nTarget asset: Windows Domain Controller\nDetection source: EDR Alert - Encoded PowerShell Commands\nInitial vector: Spearphishing attachment\nTactics: Execution (T1059.001)",
    "MITRE-2: Password spraying (T1110.003)":
        "Incident type: Password Spraying\nTarget asset: Azure AD / Microsoft 365\nDetection source: SIEM Alert - Multiple Failed Logins Across Accounts\nInitial vector: External brute force\nTactics: Credential Access (T1110.003)",

    # Asset-Aware Tests (model should reference specific hostnames/IPs)
    "Asset-1: Ransomware + endpoint assets":
        "Incident type: Ransomware\nTarget asset: Windows AD Server\nDetection source: EDR Alert - Suspicious File Encryption\nInitial vector: Email Attachment (Phishing)\nTactics and techniques:\n- Initial Access: Phishing\n- Execution: User Execution\n- Impact: Data Encrypted for Impact\nSeverity: High\nTags: malware, lateral_movement, windows, ransomware, phishing"
        + ENDPOINT_NETWORK_ASSETS,
    "Asset-2: Phishing + email/identity assets":
        "Incident type: Phishing\nTarget asset: Employee Workstations\nDetection source: Email Gateway Alert - Suspicious Attachment\nInitial vector: Spearphishing Attachment\nTactics and techniques:\n- Initial Access: Spearphishing Attachment\n- Execution: Malicious File\n- Credential Access: Input Capture\nSeverity: High\nTags: phishing, credential_theft, email"
        + EMAIL_IDENTITY_ASSETS,

    # Complex / Multi-Stage Scenarios
    "Complex-1: APT multi-stage":
        "Advanced persistent threat detected. Initial access via spearphishing, followed by persistence through scheduled tasks, lateral movement using PsExec and WMI, and staging of data in a hidden directory for exfiltration via DNS tunneling.",
    "Complex-2: Double extortion ransomware":
        "Ransomware group has encrypted critical servers AND exfiltrated 500GB of sensitive data. Ransom note threatens public data leak in 72 hours. Business operations halted.",

    # Edge Cases
    "Edge-1: IoT / OT compromise":
        "Smart building HVAC system compromised. Anomalous network traffic from HVAC controller to external C2 server. System is on same VLAN as corporate network.",
    "Edge-2: Supply chain attack":
        "Supply chain compromise discovered. A widely-used internal npm package was found to contain a backdoor injected during the CI/CD pipeline. The package is deployed across 30+ microservices.",
}

# --- Quality scoring ---
NIST_PHASES = ["Identification", "Containment", "Eradication", "Recovery", "Lessons Learned"]

GENERIC_TOOLS = {
    "incident response software", "malware removal tool", "system restore tool",
    "antivirus software", "security tool", "monitoring tool", "response tool",
    "network tool", "forensic tool", "backup tool",
}

REAL_TOOLS = {
    "crowdstrike", "falcon", "splunk", "wireshark", "velociraptor", "yara",
    "volatility", "carbon black", "sentinelone", "palo alto", "veeam",
    "wazuh", "snort", "suricata", "nessus", "qualys", "tenable",
    "cloudflare", "modsecurity", "edr", "siem", "soar", "ids", "ips",
    "firewall", "misp", "thehive", "cortex", "autopsy", "ftk", "encase",
    "elastic", "kibana", "grafana", "osquery", "zeek", "bro",
    "mimikatz", "proofpoint", "mimecast", "duo", "okta", "cyberark",
    "hashicorp vault", "aws guardduty", "azure sentinel", "google docs",
    "jira", "confluence", "pagerduty", "opsgenie", "slack",
}

ASSET_HOSTNAMES = [
    "srv-dc-01", "srv-dc-02", "fw-perimeter-01", "fw-internal-01",
    "siem-splunk-01", "edr-console", "backup-veeam-01", "ws-corp-pool",
    "srv-mail-01", "mfa-provider", "pam-vault-01", "proxy-web-01",
]

VAGUE_ACTIONS = {
    "restore systems", "conduct incident response", "remove malicious files",
    "notify management", "alert and contain", "fix the issue",
    "respond to incident", "mitigate threat",
}

ATTACKER_INDICATORS = {
    "initial compromise", "data exfiltration phase", "exploitation phase",
    "attack execution", "payload delivery", "command and control setup",
}

def score_output(text, is_asset_test=False):
    """Score a single model output against quality criteria. Returns dict of checks."""
    text_lower = text.lower()
    results = {}

    # 1. NIST phase coverage
    found_phases = [p for p in NIST_PHASES if p.lower() in text_lower]
    missing_phases = [p for p in NIST_PHASES if p.lower() not in text_lower]
    results["phases_found"] = found_phases
    results["phases_missing"] = missing_phases
    results["all_phases"] = len(missing_phases) == 0

    # 2. Real tool names (at least one)
    tools_found = [t for t in REAL_TOOLS if t in text_lower]
    generic_found = [t for t in GENERIC_TOOLS if t in text_lower]
    results["real_tools"] = tools_found
    results["generic_tools"] = generic_found
    results["has_real_tools"] = len(tools_found) > 0

    # 3. Defender perspective (no attacker phase names)
    attacker_hits = [a for a in ATTACKER_INDICATORS if a in text_lower]
    results["attacker_indicators"] = attacker_hits
    results["defender_perspective"] = len(attacker_hits) == 0

    # 4. Not truncated
    results["truncated"] = not (text.rstrip().endswith(("<|end_of_text|>", "Resolved", "resolved"))
                                 or "lessons learned" in text_lower)

    # 5. Vague actions check
    vague_hits = [v for v in VAGUE_ACTIONS if v in text_lower]
    results["vague_actions"] = vague_hits
    results["specific_actions"] = len(vague_hits) == 0

    # 6. Asset-awareness (only scored for asset-aware tests)
    assets_found = [h for h in ASSET_HOSTNAMES if h in text]
    results["assets_found"] = assets_found
    results["asset_count"] = len(assets_found)
    results["has_affected_assets"] = "affected assets:" in text_lower
    if is_asset_test:
        results["asset_aware"] = len(assets_found) >= 3  # Should reference at least 3 hostnames
    else:
        results["asset_aware"] = True  # N/A for non-asset tests

    # Overall pass/fail
    results["pass"] = all([
        results["all_phases"],
        results["has_real_tools"],
        results["defender_perspective"],
        not results["truncated"],
        results["specific_actions"],
        results["asset_aware"],
    ])

    return results

# --- Run all tests ---
print("=" * 80)
print("IRP MODEL TEST SUITE v2 -- Post-Retrain Quality Assessment")
print("=" * 80)

all_results = {}
for name, prompt_input in test_prompts.items():
    is_asset = name.startswith("Asset-")
    inputs = tokenizer(
        [alpaca_prompt.format(INSTRUCTION, prompt_input, "")],
        return_tensors="pt"
    ).to("cuda")

    out = model.generate(**inputs, max_new_tokens=512, use_cache=True)
    raw = tokenizer.batch_decode(out)[0]

    # Extract just the response portion
    response = raw.split("### Response:")[-1].strip() if "### Response:" in raw else raw

    scores = score_output(response, is_asset_test=is_asset)
    all_results[name] = {"response": response, "scores": scores, "is_asset": is_asset}

    status = "PASS" if scores["pass"] else "FAIL"
    print(f"\n{'─' * 80}")
    print(f"[{status}] {name}")
    print(f"{'─' * 80}")

    # Print compact score line
    phase_str = f"Phases: {len(scores['phases_found'])}/5"
    if scores["phases_missing"]:
        phase_str += f" (missing: {', '.join(scores['phases_missing'])})"
    tools_str = f"Tools: {', '.join(scores['real_tools'][:5])}" if scores["real_tools"] else "Tools: NONE"
    flags = []
    if not scores["defender_perspective"]:
        flags.append("ATTACKER-PERSPECTIVE")
    if scores["truncated"]:
        flags.append("TRUNCATED")
    if not scores["specific_actions"]:
        flags.append(f"VAGUE({', '.join(scores['vague_actions'])})")
    if is_asset and not scores["asset_aware"]:
        flags.append(f"NO-ASSET-REFS(found {scores['asset_count']})")
    flag_str = " | ".join(flags) if flags else "OK"

    print(f"  {phase_str}")
    print(f"  {tools_str}")
    if is_asset:
        print(f"  Assets referenced: {scores['asset_count']} -- {', '.join(scores['assets_found'][:6])}")
        print(f"  Affected Assets section: {'Yes' if scores['has_affected_assets'] else 'No'}")
    print(f"  Flags: {flag_str}")
    print(f"\n  Output preview:")
    # Print first 500 chars of response
    preview = response[:500].replace("\n", "\n  ")
    print(f"  {preview}")
    if len(response) > 500:
        print(f"  [...{len(response) - 500} more chars]")

# --- Summary scorecard ---
print(f"\n{'=' * 80}")
print("SUMMARY SCORECARD")
print(f"{'=' * 80}")

total = len(all_results)
passed = sum(1 for r in all_results.values() if r["scores"]["pass"])
all_phases_count = sum(1 for r in all_results.values() if r["scores"]["all_phases"])
real_tools_count = sum(1 for r in all_results.values() if r["scores"]["has_real_tools"])
defender_count = sum(1 for r in all_results.values() if r["scores"]["defender_perspective"])
not_truncated_count = sum(1 for r in all_results.values() if not r["scores"]["truncated"])
specific_count = sum(1 for r in all_results.values() if r["scores"]["specific_actions"])

asset_tests = {k: v for k, v in all_results.items() if v["is_asset"]}
asset_passed = sum(1 for r in asset_tests.values() if r["scores"]["asset_aware"])

print(f"  Overall Pass Rate:    {passed}/{total} ({100*passed/total:.0f}%)")
print(f"  All 5 NIST Phases:    {all_phases_count}/{total}")
print(f"  Real Tool Names:      {real_tools_count}/{total}")
print(f"  Defender Perspective:  {defender_count}/{total}")
print(f"  Not Truncated:        {not_truncated_count}/{total}")
print(f"  Specific Actions:     {specific_count}/{total}")
print(f"  Asset-Aware (>=3 refs): {asset_passed}/{len(asset_tests)} asset tests")
print(f"{'=' * 80}")

==((====))==  Unsloth 2026.3.7: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
IRP MODEL TEST SUITE v2 -- Post-Retrain Quality Assessment

────────────────────────────────────────────────────────────────────────────────
[FAIL] Core-1: Ransomware + SMB lateral
────────────────────────────────────────────────────────────────────────────────
  Phases: 5/5
  Tools: palo alto, wireshark, yara, crowdstrike, tenable
  Flags: VAGUE(restore systems)

  Output preview:
  Playbook:
  
  Phase: Identification
  Action: Analyze network traffic, monitor for suspicious SMB activity, and review logs f

In [19]:
# === BASE vs FINE-TUNED COMPARISON (for poster) ===
from unsloth import FastLanguageModel

comparison_prompt = """Incident type: Ransomware
Target asset: Windows AD Server
Detection source: EDR Alert - Suspicious File Encryption
Initial vector: Email Attachment (Phishing)
Severity: High

Organization assets:
- srv-dc-01: Dell PowerEdge R750 server, datacenter (10.20.1.10), primary domain controller
- edr-console: CrowdStrike Falcon edr, cloud, endpoint detection and response
- siem-splunk-01: Splunk Enterprise 9.2 siem, datacenter (10.20.2.10)
- fw-perimeter-01: Palo Alto PA-850 firewall, dmz (10.0.0.1)
- backup-veeam-01: Veeam Backup and Replication 12, datacenter (10.20.4.10)"""

INSTRUCTION = "Create a detailed incident response playbook for the following cyber incident."

# --- 1. Base model (no LoRA) ---
base_model, base_tok = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.1-8B-bnb-4bit",
    max_seq_length=2048, load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

base_in = base_tok(
    [alpaca_prompt.format(INSTRUCTION, comparison_prompt, "")],
    return_tensors="pt"
).to("cuda")
base_out = base_model.generate(**base_in, max_new_tokens=512, use_cache=True)
base_response = base_tok.batch_decode(base_out)[0]
if "### Response:" in base_response:
    base_response = base_response.split("### Response:")[-1].strip()
base_response = base_response.replace("<|end_of_text|>", "").strip()

del base_model  # free VRAM

print("=== BASE MODEL (Llama 3.1 8B — no fine-tuning) ===")
print(base_response)

# --- 2. Fine-tuned model (LoRA) ---
ft_model, ft_tok = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/irp_exports/ir_assistant_lora",
    max_seq_length=2048, load_in_4bit=True,
)
FastLanguageModel.for_inference(ft_model)

ft_in = ft_tok(
    [alpaca_prompt.format(INSTRUCTION, comparison_prompt, "")],
    return_tensors="pt"
).to("cuda")
ft_out = ft_model.generate(**ft_in, max_new_tokens=512, use_cache=True)
ft_response = ft_tok.batch_decode(ft_out)[0]
if "### Response:" in ft_response:
    ft_response = ft_response.split("### Response:")[-1].strip()
ft_response = ft_response.replace("<|end_of_text|>", "").strip()

del ft_model  # free VRAM

print("\n=== FINE-TUNED MODEL (LoRA on IRP dataset) ===")
print(ft_response)

# --- 3. Summary ---
nist = ["Identification", "Containment", "Eradication", "Recovery", "Lessons Learned"]
hostnames = ["srv-dc-01", "edr-console", "siem-splunk-01", "fw-perimeter-01", "backup-veeam-01"]
print("\n=== COMPARISON SUMMARY ===")
print(f"NIST Phases:  Base {len([p for p in nist if p.lower() in base_response.lower()])}/5  |  Fine-Tuned {len([p for p in nist if p.lower() in ft_response.lower()])}/5")
print(f"Asset Refs:   Base {len([h for h in hostnames if h in base_response])}/5  |  Fine-Tuned {len([h for h in hostnames if h in ft_response])}/5")


==((====))==  Unsloth 2026.3.7: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
=== BASE MODEL (Llama 3.1 8B — no fine-tuning) ===
1. Upon detection of the ransomware, the following actions should be taken:
    1. Identify the source of the infection
    2. Quarantine the affected systems
    3. Perform a full system backup
    4. Restore the system from the backup
    5. Update the anti-virus software and perform a full system scan
    6. Check for any suspicious files or processes and remove them if necessary
    7. Perform a full system audit to identify any potential vulnerabilities